<a href="https://colab.research.google.com/github/StewartIsaacs/Course-Practice/blob/main/PopulationData_Experimenting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The purpose of this notebook is for me to get some experience using Google CoLab and doing some simple API runs to download population data from the US Census most likely

Took the below example code straight from the UN website

In [ ]:
import pandas as pd
import requests
import json

# Declares the base url for calling the API
base_url = "https://population.un.org/dataportalapi/api/v1"

# Creates the target URL, indicators, in this instance
target = base_url + "/locations/"

# Get the response, which includes the first page of data as well as information on pagination and number of records
response = requests.get(target)

# Converts call into JSON
j = response.json()

# Converts JSON into a pandas DataFrame.
df = pd.json_normalize(j['data']) # pd.json_normalize flattens the JSON to accomodate nested lists within the JSON structure



In [ ]:
df

,id,name,iso3,iso2,longitude,latitude
0,4,Afghanistan,AFG,AF,67.709953,33.939110
1,8,Albania,ALB,AL,20.168331,41.153332
2,12,Algeria,DZA,DZ,1.659626,28.033886
3,16,American Samoa,ASM,AS,-170.696182,-14.306021
4,20,Andorra,AND,AD,1.521801,42.506287
...,...,...,...,...,...,...
95,364,Iran (Islamic Republic of),IRN,IR,53.688046,32.427910
96,368,Iraq,IRQ,IQ,43.679291,33.223190
97,372,Ireland,IRL,IE,-8.243890,53.412910
98,376,Israel,ISR,IL,34.851612,31.046051


In [ ]:
while j['nextPage'] != None:
    # Reset the target to the next page
    target = j['nextPage']

    #call the API for the next page
    response = requests.get(target)

    # Convert response to JSON format
    j = response.json()

    # Store the next page in a data frame
    df_temp = pd.json_normalize(j['data'])

    # Append next page to the data frame
    df = pd.concat([df,df_temp],ignore_index=False)

In [ ]:
df

,id,name,iso3,iso2,longitude,latitude
0,4,Afghanistan,AFG,AF,67.709953,33.939110
1,8,Albania,ALB,AL,20.168331,41.153332
2,12,Algeria,DZA,DZ,1.659626,28.033886
3,16,American Samoa,ASM,AS,-170.696182,-14.306021
4,20,Andorra,AND,AD,1.521801,42.506287
...,...,...,...,...,...,...
95,5501,Southern Asia,SAS,8S,NaN,NaN
96,5502,"Europe, Northern America, Australia and New Ze...",SDG,SD,NaN,NaN
97,5503,High-and-upper-middle-income countries,HUM,WB,NaN,NaN
98,5504,Low-and-Lower-middle-income countries,LLM,WB,NaN,NaN


**Example 4: Returning data on multiple indicators and geographical areas**


Below is one more example using a more complicated search in which a user wishes to retrieve data on all family planning indicators for countries in the Western Africa region for 2020. The API is first called to retrieve a list of all geographical areas in order to identify the Western Africa id. Then, the API is called to find a complete list of indicators related to family planning. The information retrieved from the first two calls is then used to access the desired data.

In [ ]:
# Define a function that will take a relative path as an input, call the API, and return a dataframe
def callAPI(relative_path:str, topic_list:bool = False) -> pd.DataFrame:
    base_url = "https://population.un.org/dataportalapi/api/v1"
    target = base_url + relative_path # Query string parameters may be appended here or directly in the provided relative path
    # Calls the API
    response = requests.get(target)
    # Reformats response into a JSON object
    j = response.json()
    # The block below will deal with paginated results.
    # If results not paginated, this will be skipped.
    try:
      # If results are paginated, they are transformed into a python dictionary.
      # The data may be accessed using the 'data' key of the dictionary.
        df = pd.json_normalize(j['data'])
        # As long as the nextPage key of the dictionary contains an address for the next API call, the function will continue to call the API and append the results to the dataframe.
        while j['nextPage'] is not None:
            response = requests.get(j['nextPage'])
            j = response.json()
            df_temp = pd.json_normalize(j['data'])
            #df = df.append(df_temp)
            df = pd.concat([df,df_temp],ignore_index=False)

    except:
        if topic_list:
            df = pd.json_normalize(j, 'indicators')
        else:
            df = pd.DataFrame(j)
    return(df)

# Uses callAPI function to get a list of locations
df_locations = callAPI("/locationsWithAggregates/")

# Identifies ID code for Western Africa
western_africa_id = df_locations.loc[df_locations["Name"]=="Western Africa", "Id"].iloc[0]

# Restricts the dataframe to only include geographies from Western Africa
df_locations = df_locations[df_locations['parentId']==western_africa_id]

# Stores country codes in a list
country_codes = [str(code) for code in df_locations["id"].values]

# Converts country code list into a string to be used in later API call
country_selection_string = ",".join(country_codes)

# Uses callAPI function to get a list of Family Planning indicators
df_topics = callAPI("/topics/FP/indicators", topic_list=True)

# Stores indicator codes in a list
indicator_codes = [str(code) for code in df_topics["id"].values]

# Converts indicator code list into string to be used in later API call
indicator_selection_string = ",".join(indicator_codes)

# Calls the API to return the indicator values for the selected indicators and countries.
df = callAPI(f"/data/indicators/{indicator_selection_string}/locations/{country_selection_string}/start/2020/end/2020")

# Finally, filters the returned results to only include median values for All Women, and limits the number of columns retained in the new dataframe.
df2 = df.loc[(df['variant']=="Median") & (df['category']=="All women"), ['location', "indicator", "variant","category","value"]]


KeyError: 'parentId'

In [ ]:
df_locations['Region'].unique()

array(['Asia', 'Europe', 'Africa', 'Oceania',
       'Latin America and the Caribbean', 'Northern America', nan,
       'Americas'], dtype=object)

In [ ]:
western_africa_id

np.int64(914)

In [ ]:
df_locations.loc[df_locations["name"]=="Western Africa"]

,id,name,iso3,iso2,longitude,latitude
50,914,Western Africa,WAF,WA,NaN,NaN
